In [0]:
%run "./01_core_functions"

## 02. Main Ingestion Loop

Each file is processed independently inside a `try/except`:
- On success -> row count captured, appended to the Delta target, logged as `SUCCESS`.
- On failure -> full error message logged as `FAILED`, loop continues to the next file
  (one bad file never blocks the rest of the batch).

The full read path is built as `Source_Path + Source_File_Name` (control table stores them
separately). Row-count mismatch vs `File_Count` is logged as a warning but still marked
`SUCCESS`.


In [0]:
batch_id = str(uuid.uuid4())
pending_rows = get_pending_files()

print(f"Batch ID: {batch_id}")
print(f"Files to process this run: {len(pending_rows)}")

for r in pending_rows:
    full_tgt = f"{r['Target_Catalog']}.{r['Target_Schema']}.{r['Target_Table_Name']}"
    print("full_tgt:",full_tgt)
    # source path + file name
    full_source_path = r["Source_Path"].rstrip("/") + "/" + r["Source_File_Name"]

    start_time = datetime.now()
    log_record = base_log_record(r, batch_id, full_source_path)

    try:
        df = (spark.read
                    .option("header", True)
                    .option("inferSchema", True)
                    .csv(full_source_path)
            )

        row_count = df.count()

        (
            df.withColumn("loaded_at", lit(datetime.now()))
            .write.format("delta").mode("append")
            # .option("mergeSchema", "true")
            .saveAsTable(full_tgt)
        )

        end_time = datetime.now()
        status = "SUCCESS"
        error_message = None

        if r["File_Count"] is not None and row_count != r["File_Count"]:
            error_message = f"Row count mismatch: expected={r['File_Count']}, actual={row_count}"
            print(f"WARNING [{r['Source_File_Name']}]: {error_message}")

        log_record.update({
            "row_count": row_count,
            "status": status,
            "error_message": error_message,
            "executed_by":executed_by,
            "start_time": start_time,
            "end_time": end_time,
            "loaded_at": end_time, 
            "duration_sec": (end_time - start_time).total_seconds(),
        })
        write_log(log_record)
        print(f"SUCCESS [{r['Source_File_Name']}] -> {full_tgt} ({row_count} rows)")

    except Exception as e:
        end_time = datetime.now()
        log_record.update({
            "row_count": None,
            "status": "FAILED",
            "error_message": str(traceback.format_exc())[:4000],
            "executed_by":executed_by,
            "start_time": start_time,
            "end_time": end_time,
            "duration_sec": (end_time - start_time).total_seconds(),
            "loaded_at": end_time,
        })
        write_log(log_record)
        print(f"FAILED [{r['Source_File_Name']}] -> {full_tgt} : {e}")
        continue

print("Batch complete.")